# Analisis Gempa Bumi Indonesia 2015–2026

Notebook ringkasan: memuat katalog bersih, menghitung temuan utama, dan menampilkan chart kunci. Pipeline lengkap ada di `scripts/` (fetch → assemble → analyze). Semua angka di notebook ini sama dengan `findings.json`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/gempa_katalog_2015_2026.csv', parse_dates=['tanggal'])
print(df.shape)
df.head()

## 1. Seberapa sering gempa terjadi?

In [ ]:
per_tahun = df.groupby('tahun').size()
per_tahun.plot.bar(color='#1E3A8A', title='Gempa M>=4,5 per Tahun (2026 s.d. 24 Okt)')
plt.ylabel('Jumlah kejadian'); plt.show()
print('Rata-rata tahun penuh 2015-2025:', round(per_tahun.loc[2015:2025].mean(), 1))

## 2. Seberapa besar? Kelas magnitudo dan 10 terbesar

In [ ]:
print('M5+:', (df.magnitudo>=5).sum(), '| M6+:', (df.magnitudo>=6).sum(), '| M7+:', (df.magnitudo>=7).sum())
df.nlargest(10, 'magnitudo')[['tanggal','magnitudo','kedalaman_km','lokasi_usgs','wilayah']]

## 3. Kedalaman: mayoritas dangkal

In [ ]:
print(df.kelas_kedalaman.value_counts(normalize=True).mul(100).round(1))
print('Dangkal + M6+:', ((df.kelas_kedalaman=='Dangkal (<70 km)') & (df.magnitudo>=6)).sum())

## 4. Wilayah teraktif

In [ ]:
df.groupby('wilayah').agg(jumlah=('magnitudo','size'), mag_maks=('magnitudo','max')).sort_values('jumlah', ascending=False)

## 5. Peta sebaran

In [ ]:
cmap = {'Dangkal (<70 km)':'#C0392B','Menengah (70-300 km)':'#D9A441','Dalam (>300 km)':'#0E7C7B'}
fig, ax = plt.subplots(figsize=(10,5))
for kls, c in cmap.items():
    s = df[df.kelas_kedalaman==kls]
    ax.scatter(s.bujur, s.lintang, s=(s.magnitudo-4.2)**2*3, c=c, alpha=.45, label=kls)
ax.set_xlim(94,142); ax.set_ylim(-12,7); ax.grid(alpha=.3); ax.legend()
ax.set_title('Sebaran gempa: ukuran = magnitudo, warna = kelas kedalaman'); plt.show()

## 6. Snapshot BMKG (Oktober 2026)

Endpoint real-time BMKG hanya menyimpan kejadian terbaru; tabel di bawah adalah snapshot saat data diambil (24 Oktober 2026).

In [ ]:
bmkg = pd.read_csv('data/bmkg_terkini.csv')
bmkg[['tanggal','magnitudo','kedalaman','wilayah','potensi']]

Catatan metodologi lengkap (filter negara lain, label Timor Leste, aturan wilayah, kelas kedalaman) ada di `data/SOURCES.md`.